In [0]:
from azure.eventhub import EventHubProducerClient, EventData
import json

#Event Hub Config
EVENT_HUB_CONNECTION_STRING = dbutils.secrets.get(scope = "key-vault-scope", key = "eventhub-connection-string")
EVENT_HUB_NAME = "weatherstreamingeventhub"

#Initialize the Event Hub producer
producer = EventHubProducerClient.from_connection_string(conn_str=EVENT_HUB_CONNECTION_STRING, eventhub_name=EVENT_HUB_NAME)

def send_event(event):
  event_data_batch = producer.create_batch()
  event_data_batch.add(EventData(json.dumps(event)))
  producer.send_batch(event_data_batch)

event = {
  "id": "222",
  "name": "Lord",}

send_event(event)

producer.close()

## API Testing

In [0]:
import requests
import json

#Getting secret value from key vault
weather_api_key = dbutils.secrets.get(scope = "key-vault-scope", key = "weatherapikey")
location = "Chennai"

base_url = "http://api.weatherapi.com/v1"

current_weather_url = f"{base_url}/current.json"

params = {
    "key": weather_api_key,
    "q": location,
}

response = requests.get(current_weather_url, params=params)

if response.status_code == 200:
    data = response.json()
    print("Current Weather:")
    print(json.dumps(data, indent=4))
else:
    print(f"Error: {response.status_code} - {response.text}")


Current Weather:
{
    "location": {
        "name": "Chennai",
        "region": "Tamil Nadu",
        "country": "India",
        "lat": 13.0833,
        "lon": 80.2833,
        "tz_id": "Asia/Kolkata",
        "localtime_epoch": 1773298133,
        "localtime": "2026-03-12 12:18"
    },
    "current": {
        "last_updated_epoch": 1773297900,
        "last_updated": "2026-03-12 12:15",
        "temp_c": 32.4,
        "temp_f": 90.3,
        "is_day": 1,
        "condition": {
            "text": "Mist",
            "icon": "//cdn.weatherapi.com/weather/64x64/day/143.png",
            "code": 1030
        },
        "wind_mph": 11.0,
        "wind_kph": 17.6,
        "wind_degree": 116,
        "wind_dir": "ESE",
        "pressure_mb": 1011.0,
        "pressure_in": 29.85,
        "precip_mm": 0.0,
        "precip_in": 0.0,
        "humidity": 67,
        "cloud": 50,
        "feelslike_c": 38.1,
        "feelslike_f": 100.5,
        "windchill_c": 29.0,
        "windchill_f": 84.3

## Complete Code for getting weather data

In [0]:
import requests
import json

#Function to handle the API response
def handle_response(response):
  if response.status_code == 200:
    data = response.json()
    return data
  else:
    print(f"Error: {response.status_code} - {response.text}")

#Function to get current weather and air quality data
def get_current_weather(base_url, api_key, location):
    current_weather_url = f"{base_url}/current.json"
    params={
        'key': api_key,
        'q': location,
        'aqi': 'yes'
    }
    response = requests.get(current_weather_url, params=params)
    return handle_response(response)

#Function to get forecast weather data
def get_forecast_weather(base_url, api_key, location, days):
    forecast_weather_url = f"{base_url}/forecast.json"
    params={
        'key': api_key,
        'q': location,
        'days': days
    }
    response = requests.get(forecast_weather_url, params=params)
    return handle_response(response)

def get_alerts(base_url, api_key, location):
    alerts_url = f"{base_url}/alerts.json"
    params={
        'key': api_key,
        'q': location,
        'alerts': 'yes'
    }
    response = requests.get(alerts_url, params=params)
    return handle_response(response)

def flatten_data(current_weather, forecast_weather, alerts):
    location_data = current_weather.get("location", {})
    current = current_weather.get("current", {})
    condition = current.get("condition", {})
    air_quality = current.get("air_quality", {})
    forecast = forecast_weather.get("forecast", {}).get("forecastday", [])
    alert_list = alerts.get("alerts", {}).get("alert", [])

    flattened_data = {
        'name': location_data.get('name'),
        'region': location_data.get('region'),
        'country': location_data.get('country'),
        'lat': location_data.get('lat'),
        'lon': location_data.get('lon'),
        'localtime': location_data.get('localtime'),
        'temp_c': current.get('temp_c'),
        'is_day': current.get('is_day'),
        'condition_text': condition.get('text'),
        'condition_icon': condition.get('icon'),
        'wind_kph': current.get('wind_kph'),
        'wind_degree': current.get('wind_degree'),
        'wind_dir': current.get('wind_dir'),
        'pressure_in': current.get('pressure_in'),
        'precip_in': current.get('precip_in'),
        'humidity': current.get('humidity'),
        'cloud': current.get('cloud'),
        'feelslike_c': current.get('feelslike_c'),
        'uv': current.get('uv'),
        'air_quality': {
            'co': air_quality.get('co'),
            'no2': air_quality.get('no2'),
            'o3': air_quality.get('o3'),
            'so2': air_quality.get('so2'),
            'pm2_5': air_quality.get('pm2_5'),
            'pm10': air_quality.get('pm10'),
            'us-epa-index': air_quality.get('us-epa-index'),
            'gb-defra-index': air_quality.get('gb-defra-index')
        },
        'alerts': [
            {
                'headline': alert.get('headline'),
                'severity': alert.get('severity'),
                'description': alert.get('desc'),
                'instruction': alert.get('instruction')
            }
            for alert in alert_list
        ],
        'forecast': [
            {
                'date': day.get('date'),
                'maxtemp_c': day.get('day', {}).get('maxtemp_c'),
                'mintemp_c': day.get('day', {}).get('mintemp_c'),
                'condition': day.get('day', {}).get('condition', {}).get('text')
            }
            for day in forecast
        ]
    }
    return flattened_data

def fetch_weather_data():

    base_url = "http://api.weatherapi.com/v1"
    api_key = dbutils.secrets.get(scope = "key-vault-scope", key = "weatherapikey")
    location = "Chennai"

    alerts = get_alerts(base_url, api_key, location)
    current_weather = get_current_weather(base_url, api_key, location)
    forecast_weather = get_forecast_weather(base_url, api_key, location, 3)
    flattened_data = flatten_data(current_weather, forecast_weather, alerts)
    print("Weather Data:")
    print(json.dumps(flattened_data, indent=3))

fetch_weather_data()




Weather Data:
{
   "name": "Chennai",
   "region": "Tamil Nadu",
   "country": "India",
   "lat": 13.0833,
   "lon": 80.2833,
   "localtime": "2026-03-12 12:54",
   "temp_c": 32.0,
   "is_day": 1,
   "condition_text": "Mist",
   "condition_icon": "//cdn.weatherapi.com/weather/64x64/day/143.png",
   "wind_kph": 19.8,
   "wind_degree": 135,
   "wind_dir": "SE",
   "pressure_in": 29.83,
   "precip_in": 0.0,
   "humidity": 67,
   "cloud": 25,
   "feelslike_c": 37.2,
   "uv": 12.4,
   "air_quality": {
      "co": 576.85,
      "no2": 13.95,
      "o3": 58.0,
      "so2": 7.85,
      "pm2_5": 31.45,
      "pm10": 35.05,
      "us-epa-index": 2,
      "gb-defra-index": 3
   },
   "alerts": [],
   "forecast": [
      {
         "date": "2026-03-12",
         "maxtemp_c": 29.4,
         "mintemp_c": 23.1,
         "condition": "Sunny"
      },
      {
         "date": "2026-03-13",
         "maxtemp_c": 29.7,
         "mintemp_c": 24.0,
         "condition": "Sunny"
      },
      {
         "d

##Sending the complete weather data to the Event HUB

In [0]:
import requests
from azure.eventhub import EventHubProducerClient, EventData
import json

EVENT_HUB_CONNECTION_STRING = dbutils.secrets.get(scope = "key-vault-scope", key = "eventhub-connection-string")
EVENT_HUB_NAME = "weatherstreamingeventhub"

#Initialize the Event Hub producer
producer = EventHubProducerClient.from_connection_string(conn_str=EVENT_HUB_CONNECTION_STRING, eventhub_name=EVENT_HUB_NAME)

def send_event(event):
  event_data_batch = producer.create_batch()
  event_data_batch.add(EventData(json.dumps(event)))
  producer.send_batch(event_data_batch)

#Function to handle the API response
def handle_response(response):
  if response.status_code == 200:
    data = response.json()
    return data
  else:
    print(f"Error: {response.status_code} - {response.text}")

#Function to get current weather and air quality data
def get_current_weather(base_url, api_key, location):
    current_weather_url = f"{base_url}/current.json"
    params={
        'key': api_key,
        'q': location,
        'aqi': 'yes'
    }
    response = requests.get(current_weather_url, params=params)
    return handle_response(response)

#Function to get forecast weather data
def get_forecast_weather(base_url, api_key, location, days):
    forecast_weather_url = f"{base_url}/forecast.json"
    params={
        'key': api_key,
        'q': location,
        'days': days
    }
    response = requests.get(forecast_weather_url, params=params)
    return handle_response(response)

def get_alerts(base_url, api_key, location):
    alerts_url = f"{base_url}/alerts.json"
    params={
        'key': api_key,
        'q': location,
        'alerts': 'yes'
    }
    response = requests.get(alerts_url, params=params)
    return handle_response(response)

def flatten_data(current_weather, forecast_weather, alerts):
    location_data = current_weather.get("location", {})
    current = current_weather.get("current", {})
    condition = current.get("condition", {})
    air_quality = current.get("air_quality", {})
    forecast = forecast_weather.get("forecast", {}).get("forecastday", [])
    alert_list = alerts.get("alerts", {}).get("alert", [])

    flattened_data = {
        'name': location_data.get('name'),
        'region': location_data.get('region'),
        'country': location_data.get('country'),
        'lat': location_data.get('lat'),
        'lon': location_data.get('lon'),
        'localtime': location_data.get('localtime'),
        'temp_c': current.get('temp_c'),
        'is_day': current.get('is_day'),
        'condition_text': condition.get('text'),
        'condition_icon': condition.get('icon'),
        'wind_kph': current.get('wind_kph'),
        'wind_degree': current.get('wind_degree'),
        'wind_dir': current.get('wind_dir'),
        'pressure_in': current.get('pressure_in'),
        'precip_in': current.get('precip_in'),
        'humidity': current.get('humidity'),
        'cloud': current.get('cloud'),
        'feelslike_c': current.get('feelslike_c'),
        'uv': current.get('uv'),
        'air_quality': {
            'co': air_quality.get('co'),
            'no2': air_quality.get('no2'),
            'o3': air_quality.get('o3'),
            'so2': air_quality.get('so2'),
            'pm2_5': air_quality.get('pm2_5'),
            'pm10': air_quality.get('pm10'),
            'us-epa-index': air_quality.get('us-epa-index'),
            'gb-defra-index': air_quality.get('gb-defra-index')
        },
        'alerts': [
            {
                'headline': alert.get('headline'),
                'severity': alert.get('severity'),
                'description': alert.get('desc'),
                'instruction': alert.get('instruction')
            }
            for alert in alert_list
        ],
        'forecast': [
            {
                'date': day.get('date'),
                'maxtemp_c': day.get('day', {}).get('maxtemp_c'),
                'mintemp_c': day.get('day', {}).get('mintemp_c'),
                'condition': day.get('day', {}).get('condition', {}).get('text')
            }
            for day in forecast
        ]
    }
    return flattened_data

def fetch_weather_data():

    base_url = "http://api.weatherapi.com/v1"
    api_key = dbutils.secrets.get(scope = "key-vault-scope", key = "weatherapikey")
    location = "Chennai"

    alerts = get_alerts(base_url, api_key, location)
    current_weather = get_current_weather(base_url, api_key, location)
    forecast_weather = get_forecast_weather(base_url, api_key, location, 3)
    flattened_data = flatten_data(current_weather, forecast_weather, alerts)
    send_event(flattened_data)

fetch_weather_data()

##Sending the weather data in streaming fashion

In [0]:
import requests
from azure.eventhub import EventHubProducerClient, EventData
import json
from datetime import datetime, timedelta

EVENT_HUB_CONNECTION_STRING = dbutils.secrets.get(scope = "key-vault-scope", key = "eventhub-connection-string")
EVENT_HUB_NAME = "weatherstreamingeventhub"

#Initialize the Event Hub producer
producer = EventHubProducerClient.from_connection_string(conn_str=EVENT_HUB_CONNECTION_STRING, eventhub_name=EVENT_HUB_NAME)

def send_event(event):
  event_data_batch = producer.create_batch()
  event_data_batch.add(EventData(json.dumps(event)))
  producer.send_batch(event_data_batch)

#Function to handle the API response
def handle_response(response):
  if response.status_code == 200:
    data = response.json()
    return data
  else:
    print(f"Error: {response.status_code} - {response.text}")

#Function to get current weather and air quality data
def get_current_weather(base_url, api_key, location):
    current_weather_url = f"{base_url}/current.json"
    params={
        'key': api_key,
        'q': location,
        'aqi': 'yes'
    }
    response = requests.get(current_weather_url, params=params)
    return handle_response(response)

#Function to get forecast weather data
def get_forecast_weather(base_url, api_key, location, days):
    forecast_weather_url = f"{base_url}/forecast.json"
    params={
        'key': api_key,
        'q': location,
        'days': days
    }
    response = requests.get(forecast_weather_url, params=params)
    return handle_response(response)

def get_alerts(base_url, api_key, location):
    alerts_url = f"{base_url}/alerts.json"
    params={
        'key': api_key,
        'q': location,
        'alerts': 'yes'
    }
    response = requests.get(alerts_url, params=params)
    return handle_response(response)

def flatten_data(current_weather, forecast_weather, alerts):
    location_data = current_weather.get("location", {})
    current = current_weather.get("current", {})
    condition = current.get("condition", {})
    air_quality = current.get("air_quality", {})
    forecast = forecast_weather.get("forecast", {}).get("forecastday", [])
    alert_list = alerts.get("alerts", {}).get("alert", [])

    flattened_data = {
        'name': location_data.get('name'),
        'region': location_data.get('region'),
        'country': location_data.get('country'),
        'lat': location_data.get('lat'),
        'lon': location_data.get('lon'),
        'localtime': location_data.get('localtime'),
        'temp_c': current.get('temp_c'),
        'is_day': current.get('is_day'),
        'condition_text': condition.get('text'),
        'condition_icon': condition.get('icon'),
        'wind_kph': current.get('wind_kph'),
        'wind_degree': current.get('wind_degree'),
        'wind_dir': current.get('wind_dir'),
        'pressure_in': current.get('pressure_in'),
        'precip_in': current.get('precip_in'),
        'humidity': current.get('humidity'),
        'cloud': current.get('cloud'),
        'feelslike_c': current.get('feelslike_c'),
        'uv': current.get('uv'),
        'air_quality': {
            'co': air_quality.get('co'),
            'no2': air_quality.get('no2'),
            'o3': air_quality.get('o3'),
            'so2': air_quality.get('so2'),
            'pm2_5': air_quality.get('pm2_5'),
            'pm10': air_quality.get('pm10'),
            'us-epa-index': air_quality.get('us-epa-index'),
            'gb-defra-index': air_quality.get('gb-defra-index')
        },
        'alerts': [
            {
                'headline': alert.get('headline'),
                'severity': alert.get('severity'),
                'description': alert.get('desc'),
                'instruction': alert.get('instruction')
            }
            for alert in alert_list
        ],
        'forecast': [
            {
                'date': day.get('date'),
                'maxtemp_c': day.get('day', {}).get('maxtemp_c'),
                'mintemp_c': day.get('day', {}).get('mintemp_c'),
                'condition': day.get('day', {}).get('condition', {}).get('text')
            }
            for day in forecast
        ]
    }
    return flattened_data

def fetch_weather_data():

    base_url = "http://api.weatherapi.com/v1"
    api_key = dbutils.secrets.get(scope = "key-vault-scope", key = "weatherapikey")
    location = "Chennai"

    alerts = get_alerts(base_url, api_key, location)
    current_weather = get_current_weather(base_url, api_key, location)
    forecast_weather = get_forecast_weather(base_url, api_key, location, 3)
    flattened_data = flatten_data(current_weather, forecast_weather, alerts)
    return flattened_data

last_sent_time = datetime.now() - timedelta(seconds=30)

def process_batch(batch_df,batch_id):
    global last_sent_time
    try:
        if datetime.now() - last_sent_time > timedelta(seconds=30):
            weather_data = fetch_weather_data()
            send_event(weather_data)
            last_sent_time=datetime.now()
            print(f"Event sent at {datetime.now()}")
    
    except Exception as e:
        print(f"Error: {e}")
        raise e

streaming_df = spark.readStream.format("rate").option("rowsPerSecond", 1).load()
streaming_df.writeStream.foreachBatch(process_batch).start().awaitTermination()

producer.close()

Event sent at 2026-03-14 14:33:10.229158
Event sent at 2026-03-14 14:33:41.683164


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can